# Notebook 06b — Current-row prompt reconstruction + v35 smoke/quick tests

Purpose:
- Never reuse old prompt content from artifacts.
- Reconstruct every prompt from the current row's premises/question/options using the locked v35 prompt style.
- Use old artifacts only for old prediction comparison when a full normalized identity key matches.
- Hard-gate prompt/current-row identity at 100%.
- Keep v35 as the only applied symbolic correction.
- Keep MC/statement verification analysis-only unless future precision gates pass.


In [ ]:
# ============================================================
# 0. Configuration
# ============================================================
from pathlib import Path
import os, re, json, math, time, shutil, random
from collections import Counter, defaultdict

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def first_existing_path(candidates, default=None):
    for p in candidates:
        if p and Path(p).exists():
            return str(Path(p))
    return default or str(candidates[0])

# Paths
BASE_MODEL = first_existing_path([
    '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1',
    '/mnt/data/qwen3-8b',
])

# IMPORTANT: this is the actual Type1 LoRA adapter, not the v35 prediction artifact folder.
LORA_ADAPTER_CANDIDATES = [
    '/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1',
    '/kaggle/input/datasets/yixuanisthebest/v2333333/DATASET_DATA_TYPE_1',
    '/kaggle/input/datasets/yixuanisthebest/v2333333',
    '/mnt/data/DATASET_DATA_TYPE_1',
]
ADAPTER = first_existing_path(LORA_ADAPTER_CANDIDATES)

OLD_PRED_CANDIDATES = [
    '/kaggle/input/datasets/yixuanisthebest/v2333333/full_model_eval_v2_flatten_preds.json',
    '/kaggle/input/datasets/yixuanisthebest/v2333333/v35_fulltest_preds.json',
    '/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1/full_model_eval_v2_flatten_preds.json',
    '/kaggle/working/full_model_eval_v2_flatten_preds.json',
    '/mnt/data/full_model_eval_v2_flatten_preds.json',
]

DATASETS = {
    'generated_v4style_300': first_existing_path([
        '/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json',
        '/mnt/data/generated_v4style_300.json',
    ]),
    # This should point to benchmark v2.1 after the dataset is replaced/patched.
    # The notebook will audit MC ambiguity and warn if the file still behaves like old v2.
    'benchmark_v2_1000': first_existing_path([
        '/kaggle/input/datasets/nguyenminhtric/test-pipeline/benchmark_v2_1000.json',
        '/mnt/data/benchmark_v2_1000.json',
    ]),
}

# Run controls
RUN_INFERENCE = True
SMOKE_DATASET = 'generated_v4style_300'
SMOKE_N = 50
RUN_FULL_AFTER_SMOKE_PASS = False  # Set True only after smoke passes.
MAX_SAMPLES_PER_DATASET = None     # Optional debug cap for full run.
CHECKPOINT_EVERY = 25

# 06b output isolation: never reuse stale notebook-06 artifacts by accident.
RUN_TAG = '06b'
FORCE_RERUN_INFERENCE = True

# Generation config. Keep deterministic.
MAX_NEW_TOKENS = 512   # 06b fix: 192 caused truncation at 'Final' (outputs up to ~270 tokens)
DO_SAMPLE = False

# Smoke gate: new smoke should not be worse than old artifact by more than this many correct answers.
SMOKE_ALLOWED_DROP_CORRECT = 3

print('OUT_DIR =', OUT_DIR)
print('BASE_MODEL =', BASE_MODEL)
print('ADAPTER =', ADAPTER)
print('DATASETS =', DATASETS)
print('RUN_INFERENCE =', RUN_INFERENCE)
print('RUN_FULL_AFTER_SMOKE_PASS =', RUN_FULL_AFTER_SMOKE_PASS)

print('RUN_TAG =', RUN_TAG)
print('FORCE_RERUN_INFERENCE =', FORCE_RERUN_INFERENCE)


In [ ]:
# ============================================================
# 1. IO + flatten + metrics utilities
# ============================================================
LABELS = ['A','B','C','D','Yes','No','Unknown']
MC_LABELS = ['A','B','C','D']
YNU_LABELS = ['Yes','No','Unknown']

FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)


def truncate_after_first_final_answer(text):
    """Keep text through the first complete Final Answer line to prevent repeated blocks from confusing parser/eval."""
    s = str(text or '')
    m = FINAL_RE.search(s)
    if not m:
        return s
    # Keep the full line containing the first final answer.
    line_end = s.find('\n', m.end())
    if line_end == -1:
        line_end = m.end()
    return s[:line_end].strip()

def load_json(path, required=True):
    p = Path(path)
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            return json.load(f), p
    if required:
        raise FileNotFoundError(path)
    return None, None

def find_first(paths):
    for p in paths:
        if Path(p).exists(): return Path(p)
    return None

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path

def normalize_label(x):
    if x is None: return None
    s=str(x).strip()
    if not s: return None
    up=s.upper()
    if up in MC_LABELS: return up
    t=s.title()
    if t in YNU_LABELS: return t
    return None

def strict_parse_answer(text, allowed):
    """Trust explicit Final Answer only. If missing, use cautious Unknown heuristic for YNU only."""
    if not text:
        return 'UNPARSEABLE'
    hits = FINAL_RE.findall(str(text))
    if hits:
        # 06b: use the FIRST explicit Final Answer. Repeated generations can append a second block.
        lab = normalize_label(hits[0])
        return lab if lab in allowed else 'UNPARSEABLE'
    low = str(text).lower()
    # Missing-final fallback: only for YNU and only for clearly indeterminate reasoning.
    if set(allowed) == set(YNU_LABELS):
        unknown_markers = [
            'cannot conclude', 'cannot determine', 'not enough information',
            'insufficient information', 'unknown whether', 'cannot infer',
            'not necessarily true', 'does not allow us to conclude',
            'there is no premise', 'no premise states', 'no premise provides'
        ]
        if any(m in low for m in unknown_markers):
            return 'Unknown'
    return 'UNPARSEABLE'

def flatten_records(raw, dataset_name):
    rows=[]
    for rec_i, rec in enumerate(raw):
        qs = rec.get('questions') or rec.get('question') or []
        ans = rec.get('answers') or rec.get('answer') or []
        exps = rec.get('explanation') or rec.get('explanations') or []
        idxs = rec.get('idx') or []
        if isinstance(qs, str): qs=[qs]
        if isinstance(ans, str): ans=[ans]
        if isinstance(exps, str): exps=[exps]*len(qs)
        if not isinstance(idxs, list): idxs=[]
        for q_i, q in enumerate(qs):
            gold = normalize_label(ans[q_i] if q_i < len(ans) else None)
            exp = exps[q_i] if isinstance(exps, list) and q_i < len(exps) else None
            idx = idxs[q_i] if isinstance(idxs, list) and q_i < len(idxs) else []
            is_mc = gold in MC_LABELS or bool(re.search(r'\n\s*A\.', str(q)))
            rows.append({
                'dataset': dataset_name,
                'flat_id': f'{dataset_name}:{rec_i}:{q_i}',
                'rec_i': rec_i,
                'q_i': q_i,
                'idx': idx,
                'premises_FOL': rec.get('premises-FOL') or rec.get('premises_FOL') or [],
                'premises_NL': rec.get('premises-NL') or rec.get('premises_NL') or [],
                'question': q,
                'gold': gold,
                'gold_raw': ans[q_i] if q_i < len(ans) else None,
                'reference_explanation': exp,
                'is_mc': bool(is_mc),
                'is_ynu': not bool(is_mc),
            })
    return rows

def per_label_metrics(rows, pred_key):
    out = {}
    for lab in LABELS:
        tp=sum(1 for r in rows if r.get('gold')==lab and r.get(pred_key)==lab)
        fp=sum(1 for r in rows if r.get('gold')!=lab and r.get(pred_key)==lab)
        fn=sum(1 for r in rows if r.get('gold')==lab and r.get(pred_key)!=lab)
        prec=tp/(tp+fp) if tp+fp else 0.0
        rec=tp/(tp+fn) if tp+fn else 0.0
        f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
        out[lab]={'precision':prec,'recall':rec,'f1':f1,'support':sum(1 for r in rows if r.get('gold')==lab),'pred_count':sum(1 for r in rows if r.get(pred_key)==lab)}
    return out

def metrics(rows, pred_key):
    n=len(rows)
    correct=sum(1 for r in rows if r.get(pred_key)==r.get('gold'))
    pl=per_label_metrics(rows, pred_key)
    macro7=sum(pl[l]['f1'] for l in LABELS)/len(LABELS)
    weighted=sum(pl[l]['f1']*pl[l]['support'] for l in LABELS)/max(1,sum(pl[l]['support'] for l in LABELS))
    mc_macro=sum(pl[l]['f1'] for l in MC_LABELS)/4
    ynu_macro=sum(pl[l]['f1'] for l in YNU_LABELS)/3
    return {
        'n': n, 'correct': correct, 'acc': correct/n if n else 0,
        'micro_f1': correct/n if n else 0, 'macro7': macro7,
        'weighted_f1': weighted, 'mc_macro': mc_macro, 'ynu_macro': ynu_macro,
        'per_label': pl,
        'pred_distribution': dict(Counter(r.get(pred_key,'MISSING') for r in rows)),
        'gold_distribution': dict(Counter(r.get('gold') for r in rows)),
    }

print('Utilities ready')

In [ ]:
# ============================================================
# 2. Prompt restoration — 06b current-row reconstruction only
# ============================================================
# Key decision in 06b:
# - Old full prompts are NOT reused as generation content.
# - Old artifacts are used only for style reference and optional old_pred comparison when identity matches.
# - Every generation prompt is rebuilt from the CURRENT row's premises_NL + question.

old_pred_path = find_first(OLD_PRED_CANDIDATES)
old_pred_by_safe_key = {}
old_artifact_count = 0
old_identity_key_count = 0
old_prompt_style_examples = []


def _norm_text_for_key(s):
    return re.sub(r'\s+', ' ', str(s or '').strip().lower())


def _norm_list_for_key(xs):
    return tuple(_norm_text_for_key(x) for x in (xs or []))


def extract_prompt_parts(prompt):
    """Extract numbered premises and question from a prompt built in the artifact style."""
    p = str(prompt or '')
    premises = []
    m = re.search(r'Premises:\s*(.*?)(?:\n\s*Question:)', p, flags=re.S|re.I)
    if m:
        block = m.group(1).strip()
        for line in block.splitlines():
            mm = re.match(r'\s*\d+\.\s*(.*?)\s*$', line)
            if mm:
                premises.append(mm.group(1).strip())
    q = ''
    mq = re.search(r'Question:\s*(.*?)(?:\n\s*(?:Reason|Instructions:|End with|Final Answer|$))', p, flags=re.S|re.I)
    if mq:
        q = re.sub(r'\s+', ' ', mq.group(1)).strip()
    return {'premises_NL': premises, 'question': q}


def row_identity_key(row):
    """Safe identity key: premises_NL + question. flat_id/question alone are unsafe across dataset versions."""
    return (
        _norm_list_for_key(row.get('premises_NL') or []),
        _norm_text_for_key(row.get('question', '')),
    )


def old_record_identity_key(old):
    # Prefer explicit premises if present; otherwise extract from prompt.
    premises = old.get('premises_NL') or old.get('premises-NL') or old.get('premises')
    q = old.get('question') or old.get('query') or ''
    if not premises:
        p = old.get('prompt') or old.get('prompt_full') or old.get('prompt_text') or ''
        parts = extract_prompt_parts(p)
        premises = parts['premises_NL']
        q = q or parts['question']
    return (_norm_list_for_key(premises or []), _norm_text_for_key(q))

if old_pred_path:
    old_preds, _ = load_json(old_pred_path)
    old_artifact_count = len(old_preds)
    print('Loaded old artifact for safe old_pred comparison only:', old_pred_path, 'n=', len(old_preds))
    for old in old_preds:
        p = old.get('prompt') or old.get('prompt_full') or old.get('prompt_text')
        if p and len(old_prompt_style_examples) < 3:
            old_prompt_style_examples.append(p)
        k = old_record_identity_key(old)
        if k[0] and k[1]:
            old_identity_key_count += 1
            old_pred_by_safe_key[k] = old
else:
    print('WARNING: old prompt/pred artifact not found. Old comparison will be skipped.')


def canonical_prompt(row):
    allowed = 'A, B, C, or D' if row['is_mc'] else 'Yes, No, or Unknown'
    premises = row.get('premises_NL') or []
    ptxt = '\n'.join(f'{i+1}. {p}' for i, p in enumerate(premises))
    # Locked style: current-row content, old-style guards.
    return (
        'You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.\n\n'
        f'Premises:\n{ptxt}\n\n'
        f'Question:\n{row["question"]}\n\n'
        f'Reason step by step briefly, cite supporting premises if useful, and End with exactly one line: Final Answer: <{allowed}>\n'
    )


def make_prompt(row):
    # 06b: always reconstruct from the current row. Never reuse old full prompt content.
    return canonical_prompt(row)


def has_final_answer_guard(prompt):
    p = str(prompt)
    return bool(
        re.search(r'End with (?:exactly )?one line:\s*Final Answer\s*:', p, flags=re.I)
        or re.search(r'End with:\s*Final Answer\s*:', p, flags=re.I)
        or re.search(r'Final Answer\s*:\s*(?:X|<[^>]+>|Yes|No|Unknown|A|B|C|D)', p, flags=re.I)
    )


def prompt_content_match(row, prompt=None):
    p = prompt if prompt is not None else make_prompt(row)
    parts = extract_prompt_parts(p)
    expected_premises = [_norm_text_for_key(x) for x in (row.get('premises_NL') or [])]
    observed_premises = [_norm_text_for_key(x) for x in parts['premises_NL']]
    expected_question = _norm_text_for_key(row.get('question', ''))
    observed_question = _norm_text_for_key(parts['question'])
    return {
        'ok': expected_premises == observed_premises and expected_question == observed_question,
        'premises_ok': expected_premises == observed_premises,
        'question_ok': expected_question == observed_question,
        'expected_premises': expected_premises,
        'observed_premises': observed_premises,
        'expected_question': expected_question,
        'observed_question': observed_question,
    }


def prompt_sanity(rows, n=20):
    sample = rows[:n]
    for r in sample:
        p = make_prompt(r)
        assert 'Use only the given premises. Do not use outside knowledge.' in p, 'missing outside-knowledge guard'
        assert has_final_answer_guard(p), 'missing final-answer line guard'
    print(f'Prompt guard sanity passed on {len(sample)} rows. Old artifact identity keys available: {old_identity_key_count}/{old_artifact_count}.')


def prompt_current_row_identity_audit(rows, n=None, fail=True, out_name='06_prompt_current_row_identity_audit.json'):
    check_rows = rows[:n] if n else rows
    bad = []
    for r in check_rows:
        p = make_prompt(r)
        chk = prompt_content_match(r, p)
        if not chk['ok']:
            bad.append({
                'flat_id': r.get('flat_id'),
                'premises_ok': chk['premises_ok'],
                'question_ok': chk['question_ok'],
                'expected_premises_head': chk['expected_premises'][:5],
                'observed_premises_head': chk['observed_premises'][:5],
                'expected_question': chk['expected_question'],
                'observed_question': chk['observed_question'],
                'prompt_head': p[:500],
            })
    report = {
        'checked': len(check_rows),
        'matched': len(check_rows) - len(bad),
        'mismatched': len(bad),
        'match_rate': (len(check_rows) - len(bad)) / len(check_rows) if check_rows else None,
        'match_pct': round(100.0 * (len(check_rows) - len(bad)) / len(check_rows), 4) if check_rows else None,
        'examples': bad[:10],
    }
    save_json(report, OUT_DIR / out_name)
    print('Prompt/current-row identity audit:', report['matched'], '/', report['checked'], 'match_pct=', report['match_pct'])
    if fail and bad:
        raise AssertionError(f'Prompt/current-row identity mismatch: {len(bad)}/{len(check_rows)}. See {OUT_DIR/out_name}')
    return report

# Keep this name for backward compatibility, but the meaning changed in 06b.
def prompt_artifact_match_audit(rows, n=20):
    return prompt_current_row_identity_audit(rows, n=n, fail=True)


In [ ]:
# ============================================================
# 3. v35 symbolic engine: typed PROVE_NO/E1 only
# ============================================================
PRED_RE = re.compile(r'¬?([A-Z][A-Za-z0-9_]*)\(x\)')
IMP_RE = re.compile(r'∀x\s*\((¬?[A-Z][A-Za-z0-9_]*\(x\))\s*→\s*(¬?[A-Z][A-Za-z0-9_]*\(x\))\)')
UNIV_RE = re.compile(r'∀x\s*\((¬?[A-Z][A-Za-z0-9_]*)\(x\)\)')
EXIST_RE = re.compile(r'∃x\s*\((¬?[A-Z][A-Za-z0-9_]*)\(x\)\)')

def atom_name(a):
    return a.replace('(x)','').strip()

def tokenize_words(s):
    return set(re.findall(r'[a-z]+', str(s).lower()))

def pred_to_words(pred):
    parts = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', pred)
    return set(p.lower() for p in parts if len(p)>1)

def best_predicate_match(question, predicates):
    qwords = tokenize_words(question)
    # remove generic words
    stop = {'does','do','all','at','least','one','some','is','there','who','which','statement','true','based','on','the','above','premises','according','following','if','then','no','every','a','an','can','we','conclude'}
    qwords = {w for w in qwords if w not in stop}
    best=(None,0,set())
    for p in predicates:
        pw = pred_to_words(p)
        if not pw: continue
        score = len(qwords & pw)/len(pw)
        if score > best[1]: best=(p,score,pw)
    return best

def build_closure(fol):
    pos=set(); neg=set(); edges=[]
    all_preds=set()
    for s in fol:
        s=str(s).strip()
        m=UNIV_RE.match(s)
        if m:
            a=m.group(1)
            name=a[1:] if a.startswith('¬') else a
            all_preds.add(name)
            if a.startswith('¬'): neg.add(name)
            else: pos.add(name)
        m=EXIST_RE.match(s)
        if m:
            a=m.group(1); name=a[1:] if a.startswith('¬') else a
            all_preds.add(name)
        m=IMP_RE.match(s)
        if m:
            left=atom_name(m.group(1)); right=atom_name(m.group(2))
            lneg=left.startswith('¬'); rneg=right.startswith('¬')
            lp=left[1:] if lneg else left; rp=right[1:] if rneg else right
            all_preds.update([lp,rp])
            edges.append((lp,lneg,rp,rneg, s))
            # contrapositive: A->B gives ¬B->¬A, including signs.
            edges.append((rp,not rneg,lp,not lneg, 'CONTRAPOSITIVE_OF: '+s))
    changed=True; path_pos={p:'GIVEN ∀x '+p for p in pos}; path_neg={p:'GIVEN ∀x ¬'+p for p in neg}
    while changed:
        changed=False
        for lp,lneg,rp,rneg,src in edges:
            has_left = lp in (neg if lneg else pos)
            if not has_left: continue
            dest_set = neg if rneg else pos
            path_set = path_neg if rneg else path_pos
            src_path = path_neg.get(lp) if lneg else path_pos.get(lp)
            if rp not in dest_set:
                dest_set.add(rp); path_set[rp]=src if not src_path else f'{src_path} -> {src}'; changed=True
    return {'pos':pos,'neg':neg,'predicates':all_preds,'path_pos':path_pos,'path_neg':path_neg}

def question_type(q):
    low=str(q).lower()
    if re.search(r'at least one|is there some|does at least one|some ', low): return 'existential'
    if re.search(r'\ball\b|\bevery\b|do all|does all|are all|is it true that every', low): return 'universal'
    if 'statement:' in low and ' if ' in low and ' then ' in low: return 'statement_conditional'
    return 'other'

def v35_verdict(row):
    if not row.get('is_ynu'): return ('ABSTAIN', None)
    q = row.get('question','')
    qt = question_type(q)
    closure = build_closure(row.get('premises_FOL') or [])
    target, score, tw = best_predicate_match(q, closure['predicates'])
    cert = {'qtype':qt,'target':target,'target_score':score,'target_words':sorted(tw) if tw else [],
            'neg_proof_universal':False,'pos_proof_universal':False,'neg_proof_path':None,'pos_proof_path':None}
    if not target or score < 0.5:
        cert['reason']='NO_TARGET_MATCH'; return ('ABSTAIN', cert)
    neg = target in closure['neg']; pos = target in closure['pos']
    cert.update({'neg_proof_universal':neg,'pos_proof_universal':pos,
                 'neg_proof_path':closure['path_neg'].get(target), 'pos_proof_path':closure['path_pos'].get(target)})
    # E1: for existential question, universal negative proves No even under positive conflict by dataset convention/classical target.
    if qt == 'existential' and neg:
        cert['rule']='E1_FORALL_NEG_NOT_EXISTS'; return ('PROVE_NO', cert)
    # Universal negative: if asked universal/other and we prove no target, No; but abstain if positive conflict.
    if qt in ('universal','other') and neg and not pos:
        cert['rule']='TYPED_PROVE_NO_NO_POSITIVE_CONFLICT'; return ('PROVE_NO', cert)
    cert['reason']='NO_SAFE_PROVE_NO'; return ('ABSTAIN', cert)

def apply_parsefix_and_v35(rows):
    for r in rows:
        allowed = MC_LABELS if r['is_mc'] else YNU_LABELS
        r['pred_raw'] = strict_parse_answer(r.get('raw_output',''), allowed)
        r['pred_parsefix'] = r['pred_raw']
        r['pred_v35'] = r['pred_parsefix']
        r['v35_verdict'], r['v35_certificate'] = ('ABSTAIN', None)
        r['v35_changed'] = False
        if r['is_ynu'] and r['pred_parsefix'] != 'No':
            verdict, cert = v35_verdict(r)
            r['v35_verdict'], r['v35_certificate'] = verdict, cert
            if verdict == 'PROVE_NO':
                r['pred_v35'] = 'No'; r['v35_changed'] = True
    return rows

print('v35 symbolic engine ready')

In [ ]:
# ============================================================
# 3b. Benchmark MC ambiguity guard: detect old-v2 style ambiguous distractors
# ============================================================
def parse_option_texts(q):
    opts = {}
    for m in re.finditer(r'(?:^|\n)\s*([A-D])\.\s*(.*?)(?=(?:\n\s*[A-D]\.)|$)', str(q), flags=re.S):
        opts[m.group(1)] = re.sub(r'\s+', ' ', m.group(2)).strip()
    return opts

def option_target_predicate(text):
    """
    Lightweight target extraction from option text.
    This is not used for correction. It is only an ambiguity/data audit.
    """
    s = str(text)
    # "Every student completes the capstone project" -> CompletesCapstone-ish by word tokens.
    words = [w.lower() for w in re.findall(r'[A-Za-z]+', s)]
    stop = {
        'a','an','the','if','then','no','not','every','all','student','students','intern','interns',
        'course','courses','teacher','teachers','applicant','applicants','lab','member','members',
        'workshop','participant','participants','library','patron','patrons','exam','candidate','candidates',
        'based','on','above','premises','conclusion','logically','follows','who','does','do','is','are',
        'there','some','at','least','one'
    }
    content = [w for w in words if w not in stop]
    return set(content)

def option_supported_by_universal_closure(option_text, premises_fol):
    """
    Conservative audit-only support check:
    returns True if option content words strongly match a universal positive predicate in closure.
    It intentionally over-approximates to expose ambiguous MC data.
    """
    closure = build_closure(premises_fol or [])
    target_words = option_target_predicate(option_text)
    if not target_words:
        return False
    for pred in closure['pos']:
        pw = pred_to_words(pred)
        if pw and len(target_words & pw) / max(1, len(pw)) >= 0.66:
            return True
    return False

def mc_ambiguity_audit(rows):
    out = []
    for r in rows:
        if not r.get('is_mc'):
            continue
        opts = parse_option_texts(r.get('question',''))
        supported = []
        for letter, text in opts.items():
            if option_supported_by_universal_closure(text, r.get('premises_FOL') or []):
                supported.append(letter)
        if len(supported) > 1:
            out.append({
                'flat_id': r.get('flat_id'),
                'gold': r.get('gold'),
                'supported_options_audit_only': supported,
                'question': r.get('question'),
            })
    return out

def run_dataset_preflight_audits():
    audit_summary = {}
    for name, path in DATASETS.items():
        try:
            raw, _ = load_json(path)
            rows = flatten_records(raw, name)
            amb = mc_ambiguity_audit(rows)
            audit_summary[name] = {
                'path': path,
                'n_flat': len(rows),
                'n_mc': sum(r.get('is_mc') for r in rows),
                'mc_ambiguity_audit_n': len(amb),
                'warning': (
                    'MC ambiguity detected. If this is benchmark_v2_1000, verify that you are using patched v2.1, not old v2.'
                    if len(amb) else ''
                ),
                'examples': amb[:5],
            }
        except Exception as e:
            audit_summary[name] = {'path': path, 'error': repr(e)}
    save_json(audit_summary, OUT_DIR / '06_dataset_preflight_audit.json')
    print(json.dumps(audit_summary, indent=2)[:5000])
    return audit_summary

dataset_preflight_audit = run_dataset_preflight_audits()


In [ ]:
# ============================================================
# 4. Model loading + inference
# ============================================================
model = None; tok = None

def load_model_once():
    global model, tok
    if model is not None and tok is not None: return model, tok
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    print('Loading tokenizer:', ADAPTER)
    assert Path(ADAPTER).exists(), f'Adapter path does not exist: {ADAPTER}'
    assert (Path(ADAPTER) / 'adapter_config.json').exists() or (Path(ADAPTER) / 'config.json').exists(), (
        'ADAPTER does not look like a LoRA/model folder. Expected adapter_config.json or config.json. '
        f'Current ADAPTER={ADAPTER}'
    )
    tok = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print('Loading base model:', BASE_MODEL, 'dtype=', dtype, 'cuda=', torch.cuda.is_available())
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map='auto' if torch.cuda.is_available() else None, trust_remote_code=True)
    print('Loading LoRA adapter:', ADAPTER)
    model = PeftModel.from_pretrained(base, ADAPTER)
    model.eval()
    return model, tok

def generate_one(prompt):
    import torch
    m,t = load_model_once()
    inputs = t(prompt, return_tensors='pt')
    inputs = {k:v.to(m.device) for k,v in inputs.items()}
    with torch.no_grad():
        out = m.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=t.eos_token_id,
            eos_token_id=t.eos_token_id,
        )
    gen = out[0][inputs['input_ids'].shape[1]:]
    decoded = t.decode(gen, skip_special_tokens=True)
    return truncate_after_first_final_answer(decoded)

print('Model loader ready')

In [ ]:
# ============================================================
# 5. Dataset runner + smoke gate
# ============================================================
def load_dataset_rows(name):
    raw, p = load_json(DATASETS[name])
    rows = flatten_records(raw, name)
    print(name, 'flattened:', len(rows), 'gold:', Counter(r['gold'] for r in rows), 'MC:', sum(r['is_mc'] for r in rows))
    prompt_sanity(rows)
    # 06b hard identity audit on a small preview here; smoke cell below audits the full smoke slice.
    prompt_current_row_identity_audit(rows, n=20, fail=True)
    return rows

def attach_old_for_smoke(rows):
    matched = 0
    for r in rows:
        old = old_pred_by_safe_key.get(row_identity_key(r))
        if old:
            matched += 1
            # support multiple old prediction field names
            r['old_pred'] = old.get('pred_v35') or old.get('pred_selected') or old.get('pred_v32') or old.get('pred') or old.get('prediction')
        else:
            r['old_pred'] = None
    print(f'Safe old_pred identity matches: {matched}/{len(rows)}')
    return rows

def run_inference_rows(rows, dataset_name, suffix, max_samples=None):
    out_path = OUT_DIR / f'{RUN_TAG}_{dataset_name}_{suffix}_preds.json'
    work = rows[:max_samples] if max_samples else rows
    # Resume if exact output exists and length matches
    if out_path.exists() and not FORCE_RERUN_INFERENCE:
        saved, _ = load_json(out_path)
        if len(saved) == len(work) and all(x.get('generated_by') == RUN_TAG for x in saved[:3]):
            print('Reusing existing:', out_path)
            return saved
    elif out_path.exists() and FORCE_RERUN_INFERENCE:
        print('FORCE_RERUN_INFERENCE=True: ignoring existing output:', out_path)
    result=[]
    start=time.time()
    for i, r in enumerate(work):
        rr=dict(r)
        rr['generated_by'] = RUN_TAG
        rr['prompt'] = make_prompt(rr)
        if RUN_INFERENCE:
            try:
                t0=time.time(); rr['raw_output']=generate_one(rr['prompt']); rr['latency_sec']=time.time()-t0; rr['generation_error']=None
            except Exception as e:
                rr['raw_output']=''; rr['latency_sec']=None; rr['generation_error']=repr(e)
        else:
            rr['raw_output']=''; rr['latency_sec']=None; rr['generation_error']='RUN_INFERENCE_FALSE'
        result.append(rr)
        if (i+1)%CHECKPOINT_EVERY==0 or i+1==len(work):
            save_json(result, out_path)
            print(f'checkpoint {dataset_name}/{suffix}: {i+1}/{len(work)} elapsed={(time.time()-start)/60:.1f}m')
    return result

def evaluate_and_save(rows, dataset_name, suffix):
    rows = apply_parsefix_and_v35(rows)
    summary = {
        'version': 'notebook_06b_current_row_prompt_locked_v35',
        'dataset': dataset_name,
        'suffix': suffix,
        'n': len(rows), 'n_mc': sum(r['is_mc'] for r in rows), 'n_ynu': sum(r['is_ynu'] for r in rows),
        'metrics_raw': metrics(rows, 'pred_raw'),
        'metrics_parsefix': metrics(rows, 'pred_parsefix'),
        'metrics_v35': metrics(rows, 'pred_v35'),
        'v35_flips': sum(1 for r in rows if r.get('v35_changed')),
        'v35_good_flips': sum(1 for r in rows if r.get('v35_changed') and r.get('pred_v35')==r.get('gold')),
        'v35_bad_flips': sum(1 for r in rows if r.get('v35_changed') and r.get('pred_v35')!=r.get('gold')),
        'missing_final_answer': sum(1 for r in rows if not FINAL_RE.findall(str(r.get('raw_output','')))),
        'generation_errors': sum(1 for r in rows if r.get('generation_error')),
    }
    pred_path = save_json(rows, OUT_DIR / f'{RUN_TAG}_{dataset_name}_{suffix}_preds_evaluated.json')
    summary_path = save_json(summary, OUT_DIR / f'{RUN_TAG}_{dataset_name}_{suffix}_summary.json')
    # Reload verify
    reloaded, _ = load_json(pred_path)
    assert len(reloaded)==len(rows), 'reload length mismatch'
    m2 = metrics(reloaded, 'pred_v35')
    assert abs(m2['acc'] - summary['metrics_v35']['acc']) < 1e-12, 'reload metric mismatch'
    print('Saved', pred_path)
    print('Saved', summary_path)
    print('v35 acc/macro7:', summary['metrics_v35']['acc'], summary['metrics_v35']['macro7'], 'flips:', summary['v35_flips'], 'good/bad:', summary['v35_good_flips'], summary['v35_bad_flips'])
    return rows, summary

# Smoke run
smoke_rows = load_dataset_rows(SMOKE_DATASET)
# 06b: hard gate: generated prompts for the full smoke slice must match current rows exactly.
prompt_current_row_identity_audit(smoke_rows[:SMOKE_N], n=None, fail=True, out_name=f'{RUN_TAG}_smoke_prompt_identity_audit.json')
smoke_rows = attach_old_for_smoke(smoke_rows)
smoke_inferred = run_inference_rows(smoke_rows, SMOKE_DATASET, f'smoke{SMOKE_N}', max_samples=SMOKE_N)
smoke_eval, smoke_summary = evaluate_and_save(smoke_inferred, SMOKE_DATASET, f'smoke{SMOKE_N}')

# Compare against old artifact where possible
old_comparable = [r for r in smoke_eval if r.get('old_pred') in LABELS]
old_correct = sum(1 for r in old_comparable if r.get('old_pred') == r.get('gold'))
new_correct = sum(1 for r in old_comparable if r.get('pred_v35') == r.get('gold'))
smoke_pass = (not old_comparable) or (new_correct >= old_correct - SMOKE_ALLOWED_DROP_CORRECT)
smoke_gate = {
    'dataset': SMOKE_DATASET,
    'n': len(smoke_eval),
    'old_comparable': len(old_comparable),
    'old_correct': old_correct,
    'new_correct': new_correct,
    'allowed_drop': SMOKE_ALLOWED_DROP_CORRECT,
    'pass': smoke_pass,
    'new_metrics': smoke_summary['metrics_v35'],
    'prompt_current_row_identity_match_pct': 100.0,
}
save_json(smoke_gate, OUT_DIR / f'{RUN_TAG}_smoke_gate.json')
print('SMOKE GATE:', smoke_gate)
if not smoke_pass:
    raise AssertionError('Smoke gate FAILED. Do not run full inference until prompt/generation drift is fixed.')

In [ ]:
# ============================================================
# 6. Optional full-run on both datasets after smoke passes
# ============================================================
full_summaries = {'smoke_gate': smoke_gate, 'datasets': {}}
if RUN_FULL_AFTER_SMOKE_PASS:
    for name in DATASETS:
        rows = load_dataset_rows(name)
        cap = MAX_SAMPLES_PER_DATASET
        inferred = run_inference_rows(rows, name, 'full', max_samples=cap)
        eval_rows, summ = evaluate_and_save(inferred, name, 'full')
        full_summaries['datasets'][name] = summ
else:
    print('RUN_FULL_AFTER_SMOKE_PASS=False, skipping expensive full-run. Set True after smoke passes.')

save_json(full_summaries, OUT_DIR / f'{RUN_TAG}_combined_summary.json')
print('Combined summary saved:', OUT_DIR / '06_combined_summary.json')

In [ ]:
# ============================================================
# 7. Subset report + v36/v37 analysis-only helpers
# ============================================================
def has_direct_conflict(row):
    c = build_closure(row.get('premises_FOL') or [])
    return bool(c['pos'] & c['neg'])

def grammar_flags(q):
    flags=[]; s=str(q)
    if re.search(r'\ba (intern|exam|athlete|analyst|employee|applicant)\b', s, re.I): flags.append('ARTICLE_A_AN')
    if re.search(r'Do all \w+s? \w+\?', s): flags.append('POSSIBLY_UNNATURAL_QUESTION')
    return flags

def tag_rows(rows):
    for r in rows:
        r['question_type'] = question_type(r.get('question',''))
        r['has_direct_fol_conflict'] = has_direct_conflict(r)
        r['grammar_flags'] = grammar_flags(r.get('question',''))
        r['is_grammar_noisy'] = bool(r['grammar_flags'])
    return rows

def subset_metrics(rows, pred_key='pred_v35'):
    subsets = {
        'all': rows,
        'mc': [r for r in rows if r.get('is_mc')],
        'ynu': [r for r in rows if r.get('is_ynu')],
        'conflict': [r for r in rows if r.get('has_direct_fol_conflict')],
        'clean_no_conflict': [r for r in rows if not r.get('has_direct_fol_conflict')],
        'grammar_noisy': [r for r in rows if r.get('is_grammar_noisy')],
        'existential': [r for r in rows if r.get('question_type')=='existential'],
        'universal': [r for r in rows if r.get('question_type')=='universal'],
        'statement_conditional': [r for r in rows if r.get('question_type')=='statement_conditional'],
    }
    return {k: metrics(v, pred_key) if v else {'n':0} for k,v in subsets.items()}

# Simple MC/statement analysis-only placeholders: report candidates but never apply.
def mc_analysis_stub(rows):
    mc=[r for r in rows if r.get('is_mc')]
    # We intentionally do not apply. This stub counts errors by gold/pred to guide v36.
    errors=[r for r in mc if r.get('pred_v35') != r.get('gold')]
    return {
        'mode':'ANALYSIS_ONLY', 'n':len(mc), 'errors':len(errors),
        'error_gold_distribution': dict(Counter(r.get('gold') for r in errors)),
        'error_pred_distribution': dict(Counter(r.get('pred_v35') for r in errors)),
        'note':'MC correction disabled. Build polarity-aware unique-entailment prover separately; require >=0.9 precision on generated and benchmark v2.1 before apply.'
    }

def statement_analysis_stub(rows):
    st=[r for r in rows if r.get('question_type')=='statement_conditional']
    errors=[r for r in st if r.get('pred_v35') != r.get('gold')]
    return {
        'mode':'ANALYSIS_ONLY', 'n':len(st), 'errors':len(errors),
        'error_gold_distribution': dict(Counter(r.get('gold') for r in errors)),
        'error_pred_distribution': dict(Counter(r.get('pred_v35') for r in errors)),
        'note':'Statement correction disabled. Need antecedent->consequent proof and counterexample witness support before apply.'
    }

analysis_summary={}
# Analyze any evaluated full/smoke files produced by this notebook.
for path in sorted(OUT_DIR.glob('06_*_preds_evaluated.json')):
    rows,_=load_json(path)
    rows=tag_rows(rows)
    name=path.stem.replace('06_','').replace('_preds_evaluated','')
    analysis_summary[name]={
        'subset_metrics': subset_metrics(rows, 'pred_v35'),
        'mc_analysis': mc_analysis_stub(rows),
        'statement_analysis': statement_analysis_stub(rows),
    }
    save_json(analysis_summary[name], OUT_DIR / f'06_{name}_subset_and_analysis.json')

save_json(analysis_summary, OUT_DIR / f'{RUN_TAG}_combined_subset_and_analysis.json')
print('Saved analysis summaries:', OUT_DIR / f'{RUN_TAG}_combined_subset_and_analysis.json')

In [ ]:
# ============================================================
# 8. Risk report
# ============================================================
risk = {
    'version': 'notebook_06b_v2_current_row_prompt_locked_v35_quicktests',
    'artifact_risk': 'LOW_RELOADED_SAVED_PREDS_WITH_CHECKPOINTS',
    'overfit_risk': 'LOW_SYMBOLIC_RULES_NO_SWEEP_CURRENT_ROW_PROMPT_LOCKED',
    'underfit_risk': 'MC_AND_STATEMENT_REMAIN_UNAPPLIED; GENERATION_QUALITY_AND_PROMPT_REPRODUCIBILITY_MUST_BE_MONITORED',
    'label_collapse': False,
    'prompt_lock': {
        'required_lines': [
            'Use only the given premises. Do not use outside knowledge.',
            'End with one line: Final Answer: <label>.'
        ],
        'old_prompt_artifact': str(old_pred_path) if old_pred_path else None,
        'old_artifact_count': old_artifact_count,
        'old_identity_key_count': old_identity_key_count,
        'prompt_content_reuse': 'DISABLED_IN_06B_CURRENT_ROW_RECONSTRUCTION',
        'run_tag': RUN_TAG,
        'force_rerun_inference': FORCE_RERUN_INFERENCE,
    },
    'smoke_gate': smoke_gate,
    'correction_policy': {
        'v35_ynu_prove_no': 'APPLY_ONLY_WITH_PROOF_CERTIFICATE',
        'mc_option_proof': 'ANALYSIS_ONLY_DISABLED',
        'statement_form': 'ANALYSIS_ONLY_DISABLED',
    }
}
save_json(risk, OUT_DIR / f'{RUN_TAG}_risk_report.json')
print(json.dumps(risk, indent=2)[:4000])

# 06b hotfix notes

- Full old prompt reuse is disabled. Prompts are rebuilt from the current row.
- Old artifacts are used only for safe old prediction comparison via normalized `(premises_NL, question)` identity.
- Smoke run fails if prompt/current-row identity is not 100%.
- Parser now reads the first explicit `Final Answer`.
- Missing-final negative prose without explicit uncertainty remains `UNPARSEABLE`.
- API quick test normalizes a pasted `/predict` URL back to the service root.

- 06b_v2 isolates outputs with `RUN_TAG='06b'` and ignores stale `06_*` artifacts by default.


# 9. Quick JSON tests + simulation pass-rate

These cells are cheap sanity checks. They print machine-readable JSON directly and report success percentages before any expensive full-run or public submission check.

In [ ]:
# ============================================================
# 9a. Quick-test JSON helpers
# ============================================================
import json, time, os, re
from pathlib import Path

QUICK_TEST_N = 50
QUICK_API_TIMEOUT = 120

# Optional. If empty, API cell will try /kaggle/working/current_api_url.txt.
QUICK_API_BASE = os.environ.get('QUICK_API_BASE', '').strip().rstrip('/')


def qjson(obj):
    """Print direct JSON for copy/paste into logs."""
    print(json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=False))


def qpct(num, den):
    return round(100.0 * num / den, 4) if den else None


def qcase(name, ok, expected=None, observed=None, extra=None):
    return {
        'name': name,
        'ok': bool(ok),
        'expected': expected,
        'observed': observed,
        'extra': extra or {},
    }


def qsummary(name, cases, extra=None):
    total = len(cases)
    passed = sum(1 for c in cases if c.get('ok'))
    return {
        'test_suite': name,
        'total': total,
        'passed': passed,
        'failed': total - passed,
        'success_pct': qpct(passed, total),
        'cases': cases,
        'extra': extra or {},
    }

print('[OK] Quick-test helpers loaded.')

In [ ]:
# ============================================================
# 9b. Local quick simulation: prompt-lock + parser + v35 synthetic rules
# Prints direct JSON with success percentages.
# No GPU call. No vLLM call. Safe to run anytime after cells 0-4.
# ============================================================
quick_cases = []

# 1) Prompt-lock quick test on first QUICK_TEST_N rows.
try:
    _rows = load_dataset_rows(SMOKE_DATASET)[:QUICK_TEST_N]
    _prompt_pass = 0
    _prompt_details = []
    for r in _rows:
        p = make_prompt(r)
        ok_outside = 'Use only the given premises. Do not use outside knowledge.' in p
        ok_final = has_final_answer_guard(p)
        chk = prompt_content_match(r, p)
        ok = ok_outside and ok_final and chk['ok']
        _prompt_pass += int(ok)
        if not ok and len(_prompt_details) < 5:
            _prompt_details.append({
                'flat_id': r.get('flat_id'),
                'outside_guard': ok_outside,
                'final_answer_guard': ok_final,
                'content_match': chk['ok'],
                'premises_ok': chk['premises_ok'],
                'question_ok': chk['question_ok'],
                'prompt_head': p[:260],
            })
    quick_cases.append(qcase(
        'prompt_lock_first_N',
        _prompt_pass == len(_rows),
        expected={'outside_guard': True, 'final_answer_guard': True, 'content_match': True, 'n': len(_rows)},
        observed={'passed': _prompt_pass, 'success_pct': qpct(_prompt_pass, len(_rows)), 'fail_examples': _prompt_details},
    ))
except Exception as e:
    quick_cases.append(qcase('prompt_lock_first_N', False, observed={'error': repr(e)}))

# 2) Strict parser policy simulation.
parser_tests = [
    {
        'name': 'explicit_yes',
        'text': 'Reasoning...\nFinal Answer: Yes',
        'allowed': YNU_LABELS,
        'expected': 'Yes',
    },
    {
        'name': 'explicit_mc_c',
        'text': 'Analysis...\nFinal Answer: C',
        'allowed': MC_LABELS,
        'expected': 'C',
    },
    {
        'name': 'missing_final_cannot_conclude_to_unknown',
        'text': 'There is no premise that supports this, so we cannot conclude it.',
        'allowed': YNU_LABELS,
        'expected': 'Unknown',
    },
    {
        'name': 'missing_final_negative_explanation_not_parse_no',
        'text': 'No premise directly states this, but the reasoning is incomplete.',
        'allowed': YNU_LABELS,
        'expected': 'UNPARSEABLE',
    },
    {
        'name': 'missing_final_mc_unparseable',
        'text': 'I think the correct option is probably A because it sounds right.',
        'allowed': MC_LABELS,
        'expected': 'UNPARSEABLE',
    },
]
_parser_cases = []
for t in parser_tests:
    got = strict_parse_answer(t['text'], t['allowed'])
    _parser_cases.append(qcase(t['name'], got == t['expected'], expected=t['expected'], observed=got))
quick_cases.append(qcase(
    'strict_parser_policy',
    all(c['ok'] for c in _parser_cases),
    expected='all parser cases pass',
    observed=qsummary('strict_parser_cases', _parser_cases),
))

# 3) v35 synthetic simulation: E1 / universal-negative proof should override non-No predictions to No.
synthetic_rows = [
    {
        'flat_id': 'sim_e1_existential_no',
        'question': 'Does at least one intern log working hours?',
        'premises_FOL': ['∀x (¬AttendsWorkshops(x))', '∀x (LogsHours(x) → AttendsWorkshops(x))'],
        'premises_NL': ['No intern attends the workshops.', 'If an intern logs working hours, then the intern attends the workshops.'],
        'gold': 'No',
        'is_mc': False,
        'is_ynu': True,
        'raw_output': 'Reasoning says maybe.\nFinal Answer: Yes',
    },
    {
        'flat_id': 'sim_universal_no',
        'question': 'Do all students submit assignments?',
        'premises_FOL': ['∀x (¬SubmitsAssignments(x))'],
        'premises_NL': ['No student submits assignments.'],
        'gold': 'No',
        'is_mc': False,
        'is_ynu': True,
        'raw_output': 'Final Answer: Unknown',
    },
    {
        'flat_id': 'sim_unknown_abstain',
        'question': 'Does at least one student receive a certificate?',
        'premises_FOL': ['∀x (AttendsLectures(x))'],
        'premises_NL': ['Every student attends lectures.'],
        'gold': 'Unknown',
        'is_mc': False,
        'is_ynu': True,
        'raw_output': 'No premise provides enough information to decide.',
    },
    {
        'flat_id': 'sim_mc_not_changed',
        'question': 'Based on the above premises, which conclusion follows?\nA. Every student attends lectures\nB. No student attends lectures\nC. Every student receives a certificate\nD. Every student graduates',
        'premises_FOL': ['∀x (AttendsLectures(x))'],
        'premises_NL': ['Every student attends lectures.'],
        'gold': 'A',
        'is_mc': True,
        'is_ynu': False,
        'raw_output': 'Final Answer: A',
    },
]
try:
    sim_eval = apply_parsefix_and_v35([dict(r) for r in synthetic_rows])
    _sim_cases = []
    for r in sim_eval:
        _sim_cases.append(qcase(
            r['flat_id'],
            r.get('pred_v35') == r.get('gold'),
            expected=r.get('gold'),
            observed={
                'pred_raw': r.get('pred_raw'),
                'pred_parsefix': r.get('pred_parsefix'),
                'pred_v35': r.get('pred_v35'),
                'v35_changed': r.get('v35_changed'),
                'v35_verdict': r.get('v35_verdict'),
                'v35_certificate': r.get('v35_certificate'),
            },
        ))
    quick_cases.append(qcase(
        'v35_synthetic_rule_simulation',
        all(c['ok'] for c in _sim_cases),
        expected='all synthetic v35 cases pass',
        observed=qsummary('v35_synthetic_cases', _sim_cases),
    ))
except Exception as e:
    quick_cases.append(qcase('v35_synthetic_rule_simulation', False, observed={'error': repr(e)}))

quick_report = qsummary('LOCAL_QUICK_SIMULATION_NO_GPU', quick_cases, extra={
    'quick_test_n': QUICK_TEST_N,
    'smoke_dataset': SMOKE_DATASET,
    'note': 'This is a cheap local simulation. It does not call the LLM or vLLM.',
})

save_json(quick_report, OUT_DIR / f'{RUN_TAG}_quick_local_simulation_report.json')
qjson(quick_report)

In [ ]:
# ============================================================
# 9c. Quick evaluated-output report: read any existing evaluated preds and print JSON pass-rate.
# Works after smoke/full cells have created 06_*_preds_evaluated.json.
# ============================================================
output_cases = []
files = sorted(OUT_DIR.glob(f'{RUN_TAG}_*_preds_evaluated.json'))

for path in files:
    try:
        rows, _ = load_json(path)
        rows = tag_rows(rows) if 'tag_rows' in globals() else rows
        m_all = metrics(rows, 'pred_v35')
        m_raw = metrics(rows, 'pred_raw') if rows and 'pred_raw' in rows[0] else None
        mc = [r for r in rows if r.get('is_mc')]
        ynu = [r for r in rows if r.get('is_ynu')]
        changed = [r for r in rows if r.get('v35_changed')]
        case = {
            'file': str(path),
            'n': len(rows),
            'success_pct_v35': round(100 * m_all['acc'], 4),
            'acc_v35': m_all['acc'],
            'macro7_v35': m_all['macro7'],
            'acc_raw': m_raw['acc'] if m_raw else None,
            'macro7_raw': m_raw['macro7'] if m_raw else None,
            'mc_success_pct': round(100 * metrics(mc, 'pred_v35')['acc'], 4) if mc else None,
            'ynu_success_pct': round(100 * metrics(ynu, 'pred_v35')['acc'], 4) if ynu else None,
            'v35_flips': len(changed),
            'v35_good_flips': sum(1 for r in changed if r.get('pred_v35') == r.get('gold')),
            'v35_bad_flips': sum(1 for r in changed if r.get('pred_v35') != r.get('gold')),
            'missing_final_answer': sum(1 for r in rows if not FINAL_RE.findall(str(r.get('raw_output','')))),
            'generation_errors': sum(1 for r in rows if r.get('generation_error')),
            'label_gold_distribution': dict(Counter(r.get('gold') for r in rows)),
            'label_pred_distribution': dict(Counter(r.get('pred_v35') for r in rows)),
        }
        output_cases.append(qcase(path.name, True, observed=case))
    except Exception as e:
        output_cases.append(qcase(path.name, False, observed={'error': repr(e)}))

report = qsummary('EVALUATED_OUTPUT_PASS_RATE_REPORT', output_cases, extra={
    'n_files': len(files),
    'note': 'If n_files=0, run the smoke/full evaluation cells first.',
})
save_json(report, OUT_DIR / f'{RUN_TAG}_quick_evaluated_output_report.json')
qjson(report)

In [ ]:
# ============================================================
# 9d. Optional public/local API quick test: /health, /v1/models, /predict
# Prints direct JSON + success percentage.
# Set QUICK_API_BASE or create /kaggle/working/current_api_url.txt first.
# ============================================================
import requests


def _normalize_api_base(txt):
    txt = str(txt or '').strip().rstrip('/')
    # Users often paste the prediction URL. Normalize to service root.
    for suffix in ['/predict', '/v1/models', '/v1/completions', '/health']:
        if txt.endswith(suffix):
            txt = txt[:-len(suffix)]
    return txt.rstrip('/')

def _read_api_base():
    if QUICK_API_BASE:
        return _normalize_api_base(QUICK_API_BASE)
    for p in [Path('/kaggle/working/current_api_url.txt'), OUT_DIR / 'current_api_url.txt']:
        if p.exists():
            txt = _normalize_api_base(p.read_text())
            if txt:
                return txt
    return ''

api_base = _read_api_base()
api_cases = []

if not api_base:
    api_report = qsummary('OPTIONAL_API_QUICK_TEST', [qcase(
        'api_base_missing',
        False,
        expected='Set QUICK_API_BASE or write current_api_url.txt',
        observed={'QUICK_API_BASE': QUICK_API_BASE},
    )], extra={'skipped': True})
    qjson(api_report)
else:
    # health
    try:
        t0 = time.time()
        r = requests.get(api_base + '/health', timeout=20)
        ok = r.status_code == 200
        api_cases.append(qcase('GET /health', ok, expected=200, observed={'status': r.status_code, 'latency_sec': round(time.time()-t0, 4), 'text_head': r.text[:300]}))
    except Exception as e:
        api_cases.append(qcase('GET /health', False, observed={'error': repr(e)}))

    # models
    try:
        t0 = time.time()
        r = requests.get(api_base + '/v1/models', timeout=30)
        js = None
        try: js = r.json()
        except Exception: pass
        ok = r.status_code == 200 and ('data' in js if isinstance(js, dict) else True)
        api_cases.append(qcase('GET /v1/models', ok, expected='HTTP 200 with model list', observed={'status': r.status_code, 'latency_sec': round(time.time()-t0, 4), 'json': js if js is not None else r.text[:500]}))
    except Exception as e:
        api_cases.append(qcase('GET /v1/models', False, observed={'error': repr(e)}))

    # Type2 known smoke
    t2_payload = {
        'query_id': 'api_quick_type2_parallel',
        'type': 'type2',
        'query': 'Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.',
        'premises': [],
        'options': [],
    }
    try:
        t0 = time.time()
        r = requests.post(api_base + '/predict', json=t2_payload, timeout=QUICK_API_TIMEOUT)
        js = r.json()
        obj = js[0] if isinstance(js, list) and js else {}
        ans = str(obj.get('answer','')).strip()
        unit = str(obj.get('unit','')).strip()
        ok_schema = isinstance(js, list) and isinstance(obj, dict) and obj.get('query_id') == 'api_quick_type2_parallel'
        ok_answer = ans in {'5', '5.0', '5.00'} and unit == 'A'
        api_cases.append(qcase('POST /predict Type2 parallel', ok_schema and ok_answer, expected={'answer':'5', 'unit':'A'}, observed={'status': r.status_code, 'latency_sec': round(time.time()-t0, 4), 'json': js}))
    except Exception as e:
        api_cases.append(qcase('POST /predict Type2 parallel', False, observed={'error': repr(e)}))

    # Type1 cheap YNU smoke. This may call vLLM; run only outside judging.
    t1_payload = {
        'query_id': 'api_quick_type1_yes',
        'type': 'type1',
        'query': 'Does at least one student receive a scholarship?',
        'premises': ['Every student receives a scholarship.'],
        'options': ['Yes', 'No', 'Uncertain'],
    }
    try:
        t0 = time.time()
        r = requests.post(api_base + '/predict', json=t1_payload, timeout=QUICK_API_TIMEOUT)
        js = r.json()
        obj = js[0] if isinstance(js, list) and js else {}
        ok_schema = isinstance(js, list) and isinstance(obj, dict) and obj.get('query_id') == 'api_quick_type1_yes'
        ok_answer = str(obj.get('answer','')).strip() == 'Yes'
        api_cases.append(qcase('POST /predict Type1 existential Yes', ok_schema and ok_answer, expected={'answer':'Yes'}, observed={'status': r.status_code, 'latency_sec': round(time.time()-t0, 4), 'json': js}))
    except Exception as e:
        api_cases.append(qcase('POST /predict Type1 existential Yes', False, observed={'error': repr(e)}))

    api_report = qsummary('OPTIONAL_API_QUICK_TEST', api_cases, extra={
        'api_base': api_base,
        'warning': 'Do not spam /predict during official judging. Use this before submission or after judging only.',
    })
    save_json(api_report, OUT_DIR / f'{RUN_TAG}_quick_api_report.json')
    qjson(api_report)

In [ ]:
# ============================================================
# 9e. One-line combined quick status JSON
# Reads all quick reports that exist and prints a compact dashboard.
# ============================================================
quick_files = [
    OUT_DIR / f'{RUN_TAG}_quick_local_simulation_report.json',
    OUT_DIR / f'{RUN_TAG}_quick_evaluated_output_report.json',
    OUT_DIR / f'{RUN_TAG}_quick_api_report.json',
    OUT_DIR / f'{RUN_TAG}_smoke_gate.json',
    OUT_DIR / '06_dataset_preflight_audit.json',
]
combined = {}
for p in quick_files:
    if p.exists():
        try:
            data, _ = load_json(p)
            combined[p.name] = data
        except Exception as e:
            combined[p.name] = {'error': repr(e)}
    else:
        combined[p.name] = {'exists': False}

# Extract top-line status.
status = {
    'local_quick_success_pct': combined.get(f'{RUN_TAG}_quick_local_simulation_report.json', {}).get('success_pct'),
    'api_quick_success_pct': combined.get(f'{RUN_TAG}_quick_api_report.json', {}).get('success_pct'),
    'smoke_gate_pass': combined.get(f'{RUN_TAG}_smoke_gate.json', {}).get('pass'),
    'smoke_new_correct': combined.get(f'{RUN_TAG}_smoke_gate.json', {}).get('new_correct'),
    'smoke_old_correct': combined.get(f'{RUN_TAG}_smoke_gate.json', {}).get('old_correct'),
    'evaluated_files_found': combined.get(f'{RUN_TAG}_quick_evaluated_output_report.json', {}).get('extra', {}).get('n_files'),
}
qjson({'quick_dashboard': status, 'reports': combined})